In [1]:
import pandas as pd
import joblib
import os
from google.colab import drive
drive.mount('/content/drive')
ruta_train = '/content/drive/My Drive/Training_And_Testing_Dataset/smoking_prediction_processed.csv'
df_train = pd.read_csv(ruta_train)

Mounted at /content/drive


In [13]:
# ========================================================
# 04_entrenamiento_y_optimizacion - MODELO GLOBAL randomforest
# ========================================================
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

# 1. Carga y preparación
df = pd.read_csv('/content/drive/My Drive/Training_And_Testing_Dataset/smoking_prediction_processed.csv')

# Dropeamos constantes (como 'oral') para limpiar el dataset
cols_const = [c for c in df.columns if df[c].nunique() <= 1]
df = df.drop(columns=cols_const)

# 2. Definición de variables
X = df.drop(columns=['ID', 'smoking'])
y = df['smoking']

# 3. Entrenamiento Global
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
rf_global = RandomForestClassifier(n_estimators=100, random_state=42)
rf_global.fit(X_tr, y_tr)

# 4. Evaluación
f1_global = f1_score(y_te, rf_global.predict(X_te))
print(f"✅ Modelo Global entrenado. F1-Score: {f1_global:.4f}")

# 5. Guardado del modelo definitivo
ruta_modelos = '/content/drive/My Drive/Training_And_Testing_Dataset/'
os.makedirs(ruta_modelos, exist_ok=True)
joblib.dump(rf_global, os.path.join(ruta_modelos, 'rf_global_final.pkl'))

print(f"🏆 Modelo guardado en: {ruta_modelos}rf_global_final.pkl")

✅ Modelo Global entrenado. F1-Score: 0.7346
🏆 Modelo guardado en: /content/drive/My Drive/Training_And_Testing_Dataset/rf_global_final.pkl


In [5]:
# ========================================================
# 04_entrenamiento_y_optimizacion PRUEBA DE GÉNERO random forest
# ========================================================
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
# 1. Carga y preparación básica
df = pd.read_csv('/content/drive/My Drive/Training_And_Testing_Dataset/smoking_prediction_processed.csv')
# Dropeamos constantes
cols_const = [c for c in df.columns if df[c].nunique() <= 1]
df = df.drop(columns=cols_const)

# 2. Modelo Unificado (Control)
X = df.drop(columns=['ID', 'smoking'])
y = df['smoking']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
rf_global = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)
f1_global = f1_score(y_te, rf_global.predict(X_te))

# 3. Modelos Separados (Tu hipótesis)
# Suponiendo que 1.0 es Hombre y 0.0 es Mujer (según tu OrdinalEncoder)
df_hombres = df[df['gender'] == 1.0]
df_mujeres = df[df['gender'] == 0.0]

def entrenar_modelo(data):
    X = data.drop(columns=['ID', 'smoking'])
    y = data['smoking']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)
    return f1_score(y_te, rf.predict(X_te))

f1_hombres = entrenar_modelo(df_hombres)
f1_mujeres = entrenar_modelo(df_mujeres)

print(f"F1-Score Modelo Global  : {f1_global:.4f}")
print(f"F1-Score Solo Hombres   : {f1_hombres:.4f}")
print(f"F1-Score Solo Mujeres   : {f1_mujeres:.4f}")
print(f"Promedio Segmentado     : {(f1_hombres + f1_mujeres)/2:.4f}")

F1-Score Modelo Global  : 0.7346
F1-Score Solo Hombres   : 0.7386
F1-Score Solo Mujeres   : 0.2951
Promedio Segmentado     : 0.5168


In [14]:
# ========================================================
# 04_entrenamiento_y_optimizacion - MODELO XGBOOST GLOBAL
# ========================================================
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

# 2. Definición de variables
X = df.drop(columns=['ID', 'smoking'])
y = df['smoking']

# 3. Entrenamiento Global
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
xgb_global = XGBClassifier(eval_metric='logloss', random_state=42)
xgb_global.fit(X_tr, y_tr)

# 4. Evaluación
f1_global_xgb = f1_score(y_te, xgb_global.predict(X_te))
print(f"✅ XGBoost Global entrenado. F1-Score: {f1_global_xgb:.4f}")

# 5. Guardado del modelo definitivo
ruta_modelos = '/content/drive/My Drive/Training_And_Testing_Dataset/'
os.makedirs(ruta_modelos, exist_ok=True)
joblib.dump(xgb_global, os.path.join(ruta_modelos, 'xgb_global_final.pkl'))

print(f"🏆 Modelo XGBoost guardado en: {ruta_modelos}xgb_global_final.pkl")

✅ XGBoost Global entrenado. F1-Score: 0.6877
🏆 Modelo XGBoost guardado en: /content/drive/My Drive/Training_And_Testing_Dataset/xgb_global_final.pkl


In [6]:
# ========================================================
# 04_entrenamiento_y_optimizacion PRUEBA GLOBAL Y POR GENERO XGBOOST
# ========================================================
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

# 1. Modelo Global XGBoost (Control)
X = df.drop(columns=['ID', 'smoking'])
y = df['smoking']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

xgb_global = XGBClassifier(eval_metric='logloss', random_state=42).fit(X_tr, y_tr)
f1_global_xgb = f1_score(y_te, xgb_global.predict(X_te))

# 2. Modelos XGBoost Separados
df_hombres = df[df['gender'] == 1.0]
df_mujeres = df[df['gender'] == 0.0]

def entrenar_xgb(data):
    X = data.drop(columns=['ID', 'smoking'])
    y = data['smoking']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    xgb = XGBClassifier(eval_metric='logloss', random_state=42).fit(X_tr, y_tr)
    return f1_score(y_te, xgb.predict(X_te))

f1_hombres_xgb = entrenar_xgb(df_hombres)
f1_mujeres_xgb = entrenar_xgb(df_mujeres)

print(f"F1-Score Global (XGBoost) : {f1_global_xgb:.4f}")
print(f"F1-Score Hombres (XGBoost): {f1_hombres_xgb:.4f}")
print(f"F1-Score Mujeres (XGBoost): {f1_mujeres_xgb:.4f}")
print(f"Promedio Segmentado       : {(f1_hombres_xgb + f1_mujeres_xgb)/2:.4f}")

F1-Score Global (XGBoost) : 0.6877
F1-Score Hombres (XGBoost): 0.7023
F1-Score Mujeres (XGBoost): 0.2604
Promedio Segmentado       : 0.4814


In [15]:
# ========================================================
# 04_entrenamiento_y_optimizacion Adaboost
# ========================================================
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

# Definimos AdaBoost: usa árboles de decisión como "base"
# n_estimators=50 es un buen punto de partida para no sobreajustar
modelo_adaboost = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1), # Árboles débiles (stumps)
    n_estimators=100,
    random_state=42
)

# Entrenamos
modelo_adaboost.fit(X_train, y_train)

# Evaluamos
f1_ada = f1_score(y_test, modelo_adaboost.predict(X_test))
print(f"F1-Score AdaBoost: {f1_ada:.4f}")

#Asegurar ruta de guardado
ruta_modelos = '/content/drive/My Drive/Training_And_Testing_Dataset/'
os.makedirs(ruta_modelos, exist_ok=True)

#Guardar el modelo AdaBoost
ruta_ada = os.path.join(ruta_modelos, 'adaboost_final.pkl')
joblib.dump(modelo_adaboost, ruta_ada)

print(f"🏆 Modelo AdaBoost guardado exitosamente en: {ruta_ada}")

F1-Score AdaBoost: 0.6744
🏆 Modelo AdaBoost guardado exitosamente en: /content/drive/My Drive/Training_And_Testing_Dataset/adaboost_final.pkl


In [18]:
# ==========================================
# 04_entrenamiento_y_optimizacion.ipynb - Regresión Logística
# ==========================================
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# 1. Creamos un Pipeline que escala los datos ANTES de pasarlos a la regresión
pipeline_log_final = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=2000, solver='lbfgs', random_state=42))
])

# 2. Entrenamos el pipeline
pipeline_log_final.fit(X_train, y_train)

# 3. Evaluamos
f1_log_scaled = f1_score(y_test, pipeline_log_final.predict(X_test))
print(f"✅ F1-Score Regresión Logística (Escalada): {f1_log_scaled:.4f}")

#Asegurar ruta de guardado
ruta_modelos = '/content/drive/My Drive/Training_And_Testing_Dataset/'
os.makedirs(ruta_modelos, exist_ok=True)

#Guardar el modelo regresíon logistica
ruta_mrl = os.path.join(ruta_modelos, 'reglogistica_final.pkl')
joblib.dump(pipeline_log_final, ruta_mrl)

print(f"🏆 Modelo regresíon logistica guardado exitosamente en: {ruta_mrl}")

✅ F1-Score Regresión Logística (Escalada): 0.6650
🏆 Modelo regresíon logistica guardado exitosamente en: /content/drive/My Drive/Training_And_Testing_Dataset/reglogistica_final.pkl


In [19]:
# ==========================================
# 04_entrenamiento_y_optimizacion.ipynb - KNN
# ==========================================
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score

# 1. Pipeline con Escalado y KNN
# 'n_neighbors=5' es el valor por defecto, pero puedes probar
# valores impares como 7, 9 o 15 si quieres experimentar.
pipeline_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5, n_jobs=-1)) # n_jobs=-1 usa todos tus núcleos
])

# 2. Entrenamiento
pipeline_knn.fit(X_train, y_train)

# 3. Evaluación
f1_knn = f1_score(y_test, pipeline_knn.predict(X_test))
print(f"✅ KNN entrenado. F1-Score: {f1_knn:.4f}")

#Asegurar ruta de guardado
ruta_modelos = '/content/drive/My Drive/Training_And_Testing_Dataset/'
os.makedirs(ruta_modelos, exist_ok=True)

#Guardar el modelo regresíon logistica
ruta_knn = os.path.join(ruta_modelos, 'KNNm_final.pkl')
joblib.dump(pipeline_knn, ruta_knn)

print(f"🏆 Modelo regresíon logistica guardado exitosamente en: {ruta_knn}")

✅ KNN entrenado. F1-Score: 0.6183
🏆 Modelo regresíon logistica guardado exitosamente en: /content/drive/My Drive/Training_And_Testing_Dataset/KNNm_final.pkl


In [21]:
# ==========================================
# 04_entrenamiento_y_optimizacion - Gradient Boosting
# ==========================================
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import f1_score

# 1. Definimos y entrenamos el modelo
# n_estimators=100 es un estándar robusto
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
gb_model.fit(X_tr, y_tr)

# 2. Evaluamos
f1_gb = f1_score(y_te, gb_model.predict(X_te))
print(f"✅ F1-Score Gradient Boosting: {f1_gb:.4f}")

# 3. Guardamos el modelo
ruta_modelos = '/content/drive/My Drive/Training_And_Testing_Dataset/'
os.makedirs(ruta_modelos, exist_ok=True)
ruta_gb = os.path.join(ruta_modelos, 'gradient_boosting_final.pkl')

joblib.dump(gb_model, ruta_gb)
print(f"🏆 Modelo Gradient Boosting guardado exitosamente en: {ruta_gb}")

✅ F1-Score Gradient Boosting: 0.6900
🏆 Modelo Gradient Boosting guardado exitosamente en: /content/drive/My Drive/Training_And_Testing_Dataset/gradient_boosting_final.pkl


In [20]:
# ==========================================
# 04_entrenamiento_y_optimizacion.ipynb - Stacking Final
# ==========================================

from sklearn.ensemble import StackingClassifier, RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# 1. Definimos los modelos base (Nivel 0)
# Usamos los mejores: Random Forest, XGBoost y Gradient Boosting
base_models = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('xgb', XGBClassifier(eval_metric='logloss', random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42))
]

# 2. Definimos el meta-modelo (Nivel 1)
# La Regresión Logística es el meta-modelo estándar para Stacking
meta_model = LogisticRegression()

# 3. Construimos el StackingClassifier
stacking_model = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5, # Validación cruzada para evitar sobreajuste
    n_jobs=-1
)

# 4. Entrenamiento
stacking_model.fit(X_train, y_train)

# 5. Evaluación
f1_stack = f1_score(y_test, stacking_model.predict(X_test))
print(f"🏆 F1-Score del Stacking: {f1_stack:.4f}")

#Asegurar ruta de guardado
ruta_modelos = '/content/drive/My Drive/Training_And_Testing_Dataset/'
os.makedirs(ruta_modelos, exist_ok=True)

#Guardar el modelo regresíon logistica
ruta_stack = os.path.join(ruta_modelos, 'Stacking_model.pkl')
joblib.dump(stacking_model, ruta_stack)

print(f"🏆 Modelo regresíon logistica guardado exitosamente en: {ruta_stack}")

🏆 F1-Score del Stacking: 0.7325
🏆 Modelo regresíon logistica guardado exitosamente en: /content/drive/My Drive/Training_And_Testing_Dataset/Stacking_model.pkl
